# 01 — Empirical wreck age–condition calibration

Publication version using the anonymised four-column calibration dataset.


## 1. Packages and paths

In [ ]:
library(mgcv)
library(ggplot2)

fig <- function(height, width) {
  options(repr.plot.width = width, repr.plot.height = height)
}



# Find repository root robustly from either repo root or analysis/ working directory
find_project_root <- function(start = getwd()) {
  p <- normalizePath(start, mustWork = TRUE)
  candidates <- unique(c(p, dirname(p), dirname(dirname(p))))

  for (candidate in candidates) {
    if (dir.exists(file.path(candidate, "data")) &&
        dir.exists(file.path(candidate, "src"))) {
      return(normalizePath(candidate, mustWork = TRUE))
    }
  }

  stop("Could not locate publication repository root.")
}

project_root <- find_project_root()
calibration_file <- file.path(
  project_root, "data", "calibration", "wreck_calibration.csv"
)
curve_file <- file.path(project_root, "data", "gam_age_curve.csv")

if (!file.exists(calibration_file)) {
  stop("Missing calibration file: ", calibration_file)
}

cat("Project root:", project_root, "\n")
cat("Calibration input:", calibration_file, "\n")


## 2. Load and define the calibration subset

In [ ]:
df <- read.csv(calibration_file)

required <- c(
  "Obs",
  "chronological_age",
  "log_CR_physics",
  "log_CR_bio"
)

missing <- setdiff(required, names(df))
if (length(missing) > 0) {
  stop("Missing required columns: ", paste(missing, collapse = ", "))
}

cal <- df[
  complete.cases(df[, required]) &
    df$Obs > 0 &
    df$chronological_age > 0,
]

cal$Obs <- as.integer(cal$Obs)

cat("Rows in public calibration file:", nrow(df), "\n")
cat("Rows used for GAM calibration:", nrow(cal), "\n")
cat("Age range:", paste(range(cal$chronological_age), collapse = " to "), "\n")
cat("Observed classes:", paste(sort(unique(cal$Obs)), collapse = ", "), "\n")

expected_classes <- seq_len(length(unique(cal$Obs)))
if (!identical(sort(unique(cal$Obs)), expected_classes)) {
  stop("Observed Muckelroy classes are not consecutive integers starting at 1.")
}


## 3. Fit the calibration models

In [ ]:
Rcat <- length(unique(cal$Obs))

model1 <- gam(
  Obs ~
    s(chronological_age, k = 5) +
    s(log_CR_physics, k = 5) +
    s(log_CR_bio, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

model2 <- gam(
  Obs ~
    s(chronological_age, k = 5) +
    s(log_CR_physics, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

model4 <- gam(
  Obs ~ s(chronological_age, k = 5),
  family = ocat(R = Rcat),
  data = cal,
  method = "REML"
)

summary(model1)
AIC(model1, model2, model4)

gam.check(model1)
gam.check(model2)
gam.check(model4)

## 4. Compare age–condition functions

In [ ]:
age_compare <- seq(
  min(cal$chronological_age),
  max(cal$chronological_age),
  length.out = 300
)

mean_phys <- mean(cal$log_CR_physics, na.rm = TRUE)
mean_bio <- mean(cal$log_CR_bio, na.rm = TRUE)

new_m1 <- data.frame(
  chronological_age = age_compare,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

new_m2 <- data.frame(
  chronological_age = age_compare,
  log_CR_physics = mean_phys
)

new_m4 <- data.frame(
  chronological_age = age_compare
)

expected_class <- function(model, newdata) {
  p <- predict(model, newdata = newdata, type = "response")
  as.numeric(p %*% seq_len(ncol(p)))
}

e1 <- expected_class(model1, new_m1)
e2 <- expected_class(model2, new_m2)
e4 <- expected_class(model4, new_m4)

fig(6, 8)
plot(
  age_compare, e4,
  type = "l",
  lwd = 2,
  ylim = range(c(e1, e2, e4)),
  xlab = "Chronological age (years)",
  ylab = "Expected Muckelroy condition class"
)

lines(age_compare, e2, lwd = 2, lty = 2)
lines(age_compare, e1, lwd = 2, lty = 3)

legend(
  "bottomright",
  legend = c(
    "Age only",
    "Age + physical CR",
    "Age + physical + biological CR"
  ),
  lty = c(1, 2, 3),
  lwd = 2,
  bty = "n"
)

## 5. Empirical support

In [ ]:
age_quantiles <- quantile(
  cal$chronological_age,
  probs = c(0.10, 0.90, 0.95, 0.99)
)

print(age_quantiles)

support_age_max <- 125
tail_lambda0 <- 0.01

cat("Production support age:", support_age_max, "years\n")
cat("Wrecks older than support age:",
    sum(cal$chronological_age > support_age_max), "\n")

## 6. Production `gam_age_curve.csv`

This reproduces the production logic from the development notebook:

1. evaluate the full GAM at mean physical and biological deterioration pressure;
2. impose empirical monotonicity with `cummax()`;
3. use the empirical curve through 125 years;
4. beyond 125 years, continue asymptotically toward Muckelroy class 4.

The exported columns and age grid are retained exactly:
`Age`, `Raw_GAM`, `Monotonic_GAM`, `ExpectedClass`, ages 0–300.

In [ ]:
age_grid <- seq(0, 300, by = 1)

newdata <- data.frame(
  chronological_age = age_grid,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

raw_pred <- expected_class(model1, newdata)

mono_pred <- cummax(raw_pred)
model_pred <- mono_pred

support_idx <- which(age_grid <= support_age_max)
tail_idx <- which(age_grid > support_age_max)

class_at_support <- mono_pred[max(support_idx)]

model_pred[tail_idx] <- 4 - (4 - class_at_support) *
  exp(-tail_lambda0 * (age_grid[tail_idx] - support_age_max))

model_pred <- pmin(model_pred, 4)

gam_curve_out <- data.frame(
  Age = age_grid,
  Raw_GAM = raw_pred,
  Monotonic_GAM = mono_pred,
  ExpectedClass = model_pred
)

if (nrow(gam_curve_out) != 301) stop("Unexpected gam_age_curve row count.")
if (!identical(names(gam_curve_out),
               c("Age", "Raw_GAM", "Monotonic_GAM", "ExpectedClass"))) {
  stop("Unexpected gam_age_curve column structure.")
}
if (any(diff(gam_curve_out$ExpectedClass) < 0)) {
  stop("Production ExpectedClass curve is not monotonic.")
}

write.csv(
  gam_curve_out,
  file = curve_file,
  row.names = FALSE
)

cat("Saved:", curve_file, "\n")
cat("Expected class at 125 years:",
    gam_curve_out$ExpectedClass[gam_curve_out$Age == 125], "\n")

head(gam_curve_out)
tail(gam_curve_out)

## 7. Bootstrap uncertainty for the manuscript figure

This retains the bootstrap approach used in the final development figure.

In [ ]:
figure_age <- seq(
  min(cal$chronological_age, na.rm = TRUE),
  250,
  by = 1
)

figure_newdata <- data.frame(
  chronological_age = figure_age,
  log_CR_physics = mean_phys,
  log_CR_bio = mean_bio
)

figure_raw <- expected_class(model1, figure_newdata)
figure_mono <- cummax(figure_raw)
figure_model <- figure_mono

support_idx_fig <- which(figure_age <= support_age_max)
tail_idx_fig <- which(figure_age > support_age_max)

class_at_support_fig <- figure_mono[max(support_idx_fig)]

figure_model[tail_idx_fig] <- 4 - (4 - class_at_support_fig) *
  exp(-tail_lambda0 * (figure_age[tail_idx_fig] - support_age_max))

figure_model <- pmin(figure_model, 4)

curve_df <- data.frame(
  Age = figure_age,
  Raw_GAM = figure_raw,
  Monotonic_GAM = figure_mono,
  Model_curve = figure_model
)

set.seed(123)

B <- 300
pred_mat <- matrix(
  NA_real_,
  nrow = length(figure_age),
  ncol = B
)

for (b in seq_len(B)) {
  idx <- sample(seq_len(nrow(cal)), replace = TRUE)
  dat_b <- cal[idx, ]

  fit_b <- gam(
    Obs ~
      s(chronological_age, k = 5) +
      s(log_CR_physics, k = 5) +
      s(log_CR_bio, k = 5),
    family = ocat(R = Rcat),
    data = dat_b,
    method = "REML"
  )

  pred_mat[, b] <- expected_class(fit_b, figure_newdata)
}

curve_df$Raw_GAM_lwr <- apply(
  pred_mat, 1, quantile, 0.025, na.rm = TRUE
)
curve_df$Raw_GAM_upr <- apply(
  pred_mat, 1, quantile, 0.975, na.rm = TRUE
)

raw_plot_df <- curve_df[
  curve_df$Age <= max(cal$chronological_age, na.rm = TRUE),
]

## 8. Final manuscript figure

In [ ]:
#fig(7, 10)

final_figure <- ggplot() +
  geom_ribbon(
    data = raw_plot_df,
    aes(
      x = Age,
      ymin = Raw_GAM_lwr,
      ymax = Raw_GAM_upr
    ),
    alpha = 0.2
  ) +
geom_jitter(
    data = cal,
    aes(x = chronological_age, y = Obs),
    width = 0,
    height = 0.06,
    alpha = 0.45,
    color = "black",
    size = 2
  ) +
  geom_line(
    data = raw_plot_df,
    aes(x = Age, y = Raw_GAM),
    linewidth = 1,
    linetype = "dashed"
  ) +
  geom_line(
    data = curve_df,
    aes(x = Age, y = Model_curve),
    linewidth = 1.4
  ) +
  geom_vline(
    xintercept = support_age_max,
    linetype = "dotted"
  ) +
  geom_rug(
    data = cal,
    aes(x = chronological_age, y = Obs),
    sides = "b"
  ) +
  coord_cartesian(ylim = c(0.5, 4.1)) +
  xlab("Chronological age (years)") +
  ylab("Muckelroy class") +
  theme_classic() +
  theme(
    plot.title = element_blank(),
    axis.text = element_text(size = 14),
    axis.title = element_text(size = 14),
    panel.background = element_blank(),
    panel.grid = element_blank(),
    axis.line = element_line(colour = "black")
  )

final_figure

## 9. Optional figure export

The displayed figure is the manuscript figure. Export is optional and does not affect
`gam_age_curve.csv`.

In [ ]:
figure_dir <- file.path(
  project_root,
  "outputs",
  "manuscript_figures"
)

if (!dir.exists(figure_dir)) {
  dir.create(
    figure_dir,
    recursive = TRUE
  )
}

if (!dir.exists(figure_dir)) dir.create(figure_dir, recursive = TRUE)

ggsave(
  file.path(figure_dir, "gam_age_condition_calibration.pdf"),
  final_figure,
  width = 10,
  height = 7
)

ggsave(
  file.path(figure_dir, "gam_age_condition_calibration.png"),
  final_figure,
  width = 10,
  height = 7,
  dpi = 300
)

## Production outputs

The outputs that matter are:

- `data/gam_age_curve.csv`
- the final age–condition calibration figure

If calibration data or model specification change, rerun this notebook before rerunning downstream
Navigator products.